# 🚀 SSL400 Training Notebook - Fully Automated
### Run Cell 1 → Cell 2 → Cell 3 → then choose your action below

In [ ]:
# Cell 1 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted at /content/drive')

In [ ]:
# Cell 2 — SETUP: Copy everything from Google Drive to local GPU memory
import shutil, os

DRIVE_CODE    = '/content/drive/MyDrive/SSL400_Research'
DRIVE_MODELS  = '/content/drive/MyDrive/SSL400_Colab_Upload/models'
LOCAL         = '/content/ssl400'

os.makedirs(LOCAL, exist_ok=True)

# --- Copy latest code and config ---
print('Copying src and config...')
shutil.rmtree(f'{LOCAL}/src', ignore_errors=True)
shutil.copytree(f'{DRIVE_CODE}/src', f'{LOCAL}/src')

# --- Patch with the latest fixed evaluate.py ---
fixed_eval = '/content/drive/MyDrive/SSL400_Colab_Upload/src/evaluation/evaluate.py'
if os.path.exists(fixed_eval):
    shutil.copy(fixed_eval, f'{LOCAL}/src/evaluation/evaluate.py')
    print('✅ Patched evaluate.py with latest fixed version')

shutil.copy(f'{DRIVE_CODE}/config.yaml', LOCAL)

# --- Copy ALL trained models ---
print('Copying models...')
shutil.rmtree(f'{LOCAL}/models', ignore_errors=True)
if os.path.exists(DRIVE_MODELS):
    shutil.copytree(DRIVE_MODELS, f'{LOCAL}/models')
elif os.path.exists(f'{DRIVE_CODE}/models'):
    shutil.copytree(f'{DRIVE_CODE}/models', f'{LOCAL}/models')

# --- Symlink data folder (too large to copy) ---
if not os.path.exists(f'{LOCAL}/data'):
    os.symlink(f'{DRIVE_CODE}/data', f'{LOCAL}/data')

os.chdir(LOCAL)
print(f'✅ All ready! Working directory: {os.getcwd()}')
print(f'📁 Models found: {os.listdir(LOCAL+"/models") if os.path.exists(LOCAL+"/models") else "NONE - check Drive!"}')

In [ ]:
# Cell 3 — INSTALL DEPENDENCIES
!pip install tf-keras tf-models-official --quiet
print('✅ Dependencies installed')

---
## 📊 EVALUATE ALL EXPERIMENTS (Use this if all models are already trained)
Run Cell E1 to get metrics for all 4 experiments at once.

In [ ]:
# Cell E1 — EVALUATE ALL 4 EXPERIMENTS (Generates all metrics + saves to Drive)
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!python /content/ssl400/src/evaluation/evaluate.py --all

In [ ]:
# Cell E2 — EVALUATE A SINGLE EXPERIMENT (change EXP_ID as needed)
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
EXP_ID = 4   # ← Change to 1, 2, 3, or 4
!python /content/ssl400/src/evaluation/evaluate.py --exp_id={EXP_ID}

---
## 🏋️ TRAINING (Use this to train a new experiment)
Set EXP_ID in Cell T1, then run T1 → T2 → T3 in order.

In [ ]:
# Cell T1 — EXPERIMENT CONFIGURATION (set before training)
EXP_ID = 1              # ← Change to 1, 2, 3, or 4
BATCH_SIZE = 4          # Keep at 4 to prevent Out-of-Memory on Tesla T4
DRIVE_MODEL_DIR = f'/content/drive/MyDrive/SSL400_Colab_Upload/models/experiment_{EXP_ID}'
print(f'✅ Ready! EXP_ID={EXP_ID} | Drive backup: {DRIVE_MODEL_DIR}')

In [ ]:
# Cell T2 — PRE-PROCESS RAW VIDEOS INTO .NPY FRAMES
!pip install ultralytics --quiet
!python /content/ssl400/src/data/video_to_frames.py --exp_id {EXP_ID}

In [ ]:
# Cell T3 — RUN TRAINING (auto-resumes from checkpoint if session crashed)
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
!python /content/ssl400/src/training/train.py --exp_id={EXP_ID} --batch_size={BATCH_SIZE} --drive_dir={DRIVE_MODEL_DIR}

---
## 📈 VISUALIZE TRAINING CURVES

In [ ]:
# Cell V1 — SHOW TRAINING CURVES FOR ONE EXPERIMENT
import pandas as pd
import matplotlib.pyplot as plt
import os

EXP_ID = 1  # ← Change to 1, 2, 3, or 4
log_file = f'/content/ssl400/logs/experiment_{EXP_ID}/training_log_phase2.csv'

if os.path.exists(log_file):
    df = pd.read_csv(log_file)
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(df['accuracy'], label='Train Acc', linewidth=2, color='#2ca02c')
    ax1.plot(df['val_accuracy'], label='Val Acc', linewidth=2, color='#d62728', linestyle='--')
    ax1.set_title(f'Accuracy Curve - Experiment {EXP_ID}', fontsize=14, pad=15)
    ax1.legend()
    ax2.plot(df['loss'], label='Train Loss', linewidth=2, color='#1f77b4')
    ax2.plot(df['val_loss'], label='Val Loss', linewidth=2, color='#ff7f0e', linestyle='--')
    ax2.set_title(f'Loss Curve - Experiment {EXP_ID}', fontsize=14, pad=15)
    ax2.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'No training log found for EXP{EXP_ID}. Run training first!')

In [ ]:
# Cell V2 — SHOW COMPARISON TABLE (reads saved metrics JSON files)
import json, os
import pandas as pd

metrics_dir = '/content/ssl400/results/metrics'
rows = []
for exp_id in [1, 2, 3, 4]:
    fpath = f'{metrics_dir}/experiment_{exp_id}_metrics.json'
    if os.path.exists(fpath):
        with open(fpath) as f:
            d = json.load(f)
        rows.append({
            'EXP': f'EXP{exp_id}',
            'Name': d.get('exp_name', ''),
            'Top-1 Acc (%)': f"{d['top1_accuracy']*100:.2f}",
            'Top-5 Acc (%)': f"{d['top5_accuracy']*100:.2f}",
            'Macro F1': f"{d['macro_f1']:.4f}",
            'Precision': f"{d['macro_precision']:.4f}",
            'Recall': f"{d['macro_recall']:.4f}",
            'Latency (ms)': f"{d['inference_latency_ms']:.1f}"
        })
    else:
        print(f'⚠️  No metrics found for EXP{exp_id} - run Cell E1 first!')

if rows:
    df = pd.DataFrame(rows)
    print('\n📊 RESEARCH COMPARISON TABLE')
    print('='*80)
    print(df.to_string(index=False))

In [ ]:
# Cell V3 — PLOT PER-CLASS ACCURACY BAR CHART
import json, os
import matplotlib.pyplot as plt
import seaborn as sns

EXP_ID = 2  # ← Change this to the experiment you want to visualize
report_path = f'/content/ssl400/results/metrics/experiment_{EXP_ID}_classification_report.json'

if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        report = json.load(f)
    
    class_names = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg']]
    accuracies = [report[k]['f1-score'] * 100 for k in class_names]
    
    plt.figure(figsize=(10, 6))
    bars = sns.barplot(x=class_names, y=accuracies, palette='viridis')
    plt.title(f'Per-Class F1-Score (Accuracy) - Experiment {EXP_ID}', fontsize=16, pad=15)
    plt.ylabel('F1-Score (%)', fontsize=14)
    plt.xlabel('Sign Class', fontsize=14)
    plt.ylim(0, 100)
    
    for bar in bars.patches:
        plt.annotate(f'{bar.get_height():.1f}%',
                     (bar.get_x() + bar.get_width() / 2, bar.get_height() + 2),
                     ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
else:
    print(f'⚠️ Classification report not found for EXP{EXP_ID}. Run Cell E1 first!')
